In [1]:
# 2.1 Import Library dan Konfigurasi Path
import pandas as pd
import numpy as np
import os

# Konfigurasi path
DATA_PATH = '../../../no_stemming/data_preprocessing_final_no_stem.csv'
SLA_LEXICON_PATH = '../../../no_stemming/outputs/SLA/sla_lexicon_adapted.csv' # Memuat leksikon yang sudah dibuat
POLITIK_PATH = '../../../kamus/inset_vader_political_modified.csv'
OUTPUT_DIR = '../../../no_stemming/outputs/SLA'

os.makedirs(OUTPUT_DIR, exist_ok=True)
print("[INFO] Library dan konfigurasi path berhasil dimuat.")

[INFO] Library dan konfigurasi path berhasil dimuat.


In [2]:
# 2.2 Load Data Preprocessing Final
df = pd.read_csv(DATA_PATH)

print(f"\nData preprocessing berhasil dimuat: {len(df)} tweet")
print(f"Kolom: {df.columns.tolist()}")
df.head()


Data preprocessing berhasil dimuat: 13192 tweet
Kolom: ['no', 'timestamp', 'teks', 'teks_processed']


,no,timestamp,teks,teks_processed
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara ...,ADIL loh untuk yang punya kebijakan publik neg...
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan S...,tertibkan media online DPR pemerintah jangan s...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa truta...,harus dievaluasi lagi kebijakan bebas visa ter...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin ...,kebebasan bersuara berpendapat memang dijamin ...


In [3]:
# 2.3 Load Leksikon (InSet SLA + Politik SLA)

# 1. Load Leksikon SLA Adaptasi
df_inset_sla = pd.read_csv(SLA_LEXICON_PATH)
df_inset_sla['kata'] = df_inset_sla['kata'].astype(str).str.strip().str.lower()

# 2. Load Leksikon Politik
df_politik_sla = pd.read_csv(POLITIK_PATH)
df_politik_sla['kata'] = df_politik_sla['kata'].astype(str).str.strip().str.lower()

# 3. Gabungkan Kedua DataFrame
df_combined = pd.concat([df_inset_sla, df_politik_sla]).drop_duplicates(subset='kata', keep='last')

# 4. Buat Dictionary untuk Matching
sla_dict = dict(zip(df_combined['kata'], df_combined['mean']))

print(f"[INFO] Leksikon Berhasil Digabung.")
print(f"       - InSet SLA   : {len(df_inset_sla)} entri")
print(f"       - Politik     : {len(df_politik_sla)} entri")
print(f"       - Total Gabung: {len(sla_dict)} entri unik")

[INFO] Leksikon Berhasil Digabung.
       - InSet SLA   : 9074 entri
       - Politik     : 49 entri
       - Total Gabung: 9101 entri unik


In [4]:
# 2.4 Definisi Kategori Kata Fungsi
NEGASI_DAN_MODAL = {
    'tidak', 'bukan', 'jangan', 'belum', 'sangat', 'harus', 'wajib',
    'akan', 'sudah', 'sedang', 'telah', 'boleh', 'bisa'
}
KATA_HUBUNG_PREPOSISI = {
    'dan', 'atau', 'tetapi', 'karena', 'jika', 'di', 'ke', 'dari',
    'pada', 'untuk', 'dengan', 'oleh', 'hingga', 'sejak'
}
PRONOMINA_DEMONSTRATIVA = {
    'saya', 'aku', 'dia', 'kami', 'kamu', 'anda', 'ini', 'itu', 'yang'
}
PARTIKEL_KATA_TANYA = {
    'pun', 'sih', 'ya', 'lah', 'kah', 'apa', 'siapa', 'bagaimana'
}

ALL_FUNCTION_WORDS = (
    NEGASI_DAN_MODAL
    | KATA_HUBUNG_PREPOSISI
    | PRONOMINA_DEMONSTRATIVA
    | PARTIKEL_KATA_TANYA
)

In [5]:
# 2.4 Konfigurasi Remove Set (Untuk Skenario Hapus Fungsi)
REMOVE_SET = KATA_HUBUNG_PREPOSISI | PRONOMINA_DEMONSTRATIVA | PARTIKEL_KATA_TANYA

# Menggunakan df_combined agar mencakup kata fungsi dari InSet maupun Politik
df_found = df_combined[df_combined['kata'].isin(ALL_FUNCTION_WORDS)].copy().sort_values('kata')
total_fw_in_lexicon = len(df_found)

# Ambil kata fungsi yang akan dihapus secara fisik
remove_set_final = set(df_found[df_found['kata'].isin(REMOVE_SET)]['kata'])

# Gunakan dictionary gabungan 
final_dict = sla_dict 

print(f"[DIAGNOSTIK] Total kata fungsi ditemukan di Leksikon Gabungan: {total_fw_in_lexicon}")
print(f"\n[KONFIGURASI] Dictionary Gabungan siap: {len(final_dict)} entri.")
print(f"Kata fungsi yang akan dihapus secara fisik dari teks: {len(remove_set_final)} kata.")

# Menampilkan daftar kata yang akan dihapus fisik (Opsional untuk cek)
print("\n[DAFTAR] Contoh kata yang akan dihapus:")
print(sorted(list(remove_set_final))[:10]) # Tampilkan 10 saja agar tidak penuh

[DIAGNOSTIK] Total kata fungsi ditemukan di Leksikon Gabungan: 21

[KONFIGURASI] Dictionary Gabungan siap: 9101 entri.
Kata fungsi yang akan dihapus secara fisik dari teks: 13 kata.

[DAFTAR] Contoh kata yang akan dihapus:
['aku', 'anda', 'apa', 'dari', 'dia', 'itu', 'karena', 'pada', 'pun', 'saya']


In [6]:
# 2.6 Fungsi Tokenisasi
def tokenize(text):
    if not isinstance(text, str):
        return []
    return text.split()

df['tokens'] = df['teks_processed'].apply(tokenize)
print(f"\n[INFO] Tokenisasi selesai. Total token: {df['tokens'].str.len().sum():,}")


[INFO] Tokenisasi selesai. Total token: 235,559


In [7]:
# 2.7 Fungsi Lexicon Matching dengan Penghapusan Kata Fungsi
def match_lexicon_sla_remove(tokens, lexicon, remove_set):
    matched, unmatched, removed = [], [], []
    for token in tokens:
        t_low = token.lower()
        if t_low in remove_set: removed.append(token)
        elif t_low in lexicon: matched.append(token)
        else: unmatched.append(token)
    return matched, unmatched, removed

# Terapkan ke DataFrame
df[['matched_words', 'unmatched_words', 'removed_words']] = pd.DataFrame(
    df['tokens'].apply(lambda x: match_lexicon_sla_remove(x, sla_dict, REMOVE_SET)).tolist(),
    index=df.index
)

In [8]:
# 2.8 Penerapan Lexicon Matching
print("\n[PROSES] Menjalankan lexicon matching SLA dengan penghapusan fungsi...")

df[['matched_words', 'removed_words', 'unmatched_words']] = pd.DataFrame(
    df['tokens'].apply(lambda x: match_lexicon_sla_remove(x, sla_dict, REMOVE_SET)).tolist(),
    index=df.index
)

print("[INFO] Lexicon matching selesai.")


[PROSES] Menjalankan lexicon matching SLA dengan penghapusan fungsi...
[INFO] Lexicon matching selesai.


In [9]:
# 2.9 Perhitungan Statistik
total_words = df['tokens'].str.len().sum()
total_matched = df['matched_words'].str.len().sum()
total_removed = df['removed_words'].str.len().sum()
total_unmatched = df['unmatched_words'].str.len().sum()

# Token sisa setelah penghapusan (konten + negasi/modal)
filtered_words = total_matched + total_unmatched

print("\n[STATISTIK] Hasil Lexicon Matching (SLA + Hapus Fungsi):")
print(f"Total token awal         : {total_words:,}")
print(f"Dihapus (kata fungsi)    : {total_removed:,} ({(total_removed/total_words)*100:.2f}%)")
print(f"Token sisa (konten)      : {filtered_words:,}")
print(f"Matched di SLA           : {total_matched:,} ({(total_matched/filtered_words)*100:.2f}% dari token sisa)")
print(f"Unmatched                : {total_unmatched:,} ({(total_unmatched/filtered_words)*100:.2f}% dari token sisa)")
print(f"Coverage Rate (konten)   : {(total_matched/filtered_words)*100:.2f}%")


[STATISTIK] Hasil Lexicon Matching (SLA + Hapus Fungsi):
Total token awal         : 235,559
Dihapus (kata fungsi)    : 113,111 (48.02%)
Token sisa (konten)      : 122,448
Matched di SLA           : 98,456 (80.41% dari token sisa)
Unmatched                : 23,992 (19.59% dari token sisa)
Coverage Rate (konten)   : 80.41%


In [10]:
# 2.10 Preview Hasil Matching
print("\n[PREVIEW] 3 Tweet Pertama:")
for i in range(3):
    print(f"\nTweet {i+1}: {df['teks_processed'].iloc[i][:80]}...")
    print(f"  Matched  : {df['matched_words'].iloc[i][:5]}")
    print(f"  Removed  : {df['removed_words'].iloc[i][:5]}")
    print(f"  Unmatched: {df['unmatched_words'].iloc[i][:5]}")


[PREVIEW] 3 Tweet Pertama:

Tweet 1: ADIL loh untuk yang punya kebijakan publik negara ingat yang ini ! !...
  Matched  : ['ADIL', 'punya', 'kebijakan', 'ingat']
  Removed  : ['loh', 'publik', 'negara', '!', '!']
  Unmatched: ['untuk', 'yang', 'yang', 'ini']

Tweet 2: tertibkan media online DPR pemerintah jangan sporadis apalagi selektif hanya kep...
  Matched  : ['DPR', 'pemerintah', 'jangan', 'sporadis', 'selektif']
  Removed  : ['tertibkan', 'media', 'online', 'apalagi', 'kepada']
  Unmatched: ['yang']

Tweet 3: harus dievaluasi lagi kebijakan bebas visa terutama untuk negara tiongkok pak ! ...
  Matched  : ['harus', 'lagi', 'kebijakan', 'bebas', 'terutama']
  Removed  : ['dievaluasi', 'visa', 'negara', 'tiongkok', 'pak']
  Unmatched: ['untuk']


In [11]:
# 2.11 Simpan Output
output_path = os.path.join(OUTPUT_DIR, 'lexicon_matching_remove_func.csv')
df.to_csv(output_path, index=False)
print(f"\n[OUTPUT] Data berhasil disimpan ke: {output_path}")


[OUTPUT] Data berhasil disimpan ke: ../../../no_stemming/outputs/SLA\lexicon_matching_remove_func.csv
